In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import patcher
import xarray as xr
from pathlib import Path
from datetime import timedelta
from collections import defaultdict
from matplotlib import pyplot as plt
from tqdm.auto import tqdm

BASE_PATH = '../data/snow_modeling'
BASE_SCORE_PATH = '../data/score/swe'
BASE_MODEL_PATH = '../experiments/snow_modeling'

# Датасет

In [ ]:
TEST_DATES = pd.date_range('2021-06-01', '2026-03-01', freq='MS').strftime('%Y%m%d')
TRAIN_DATES = pd.date_range('1980-06-01', '2020-12-01', freq='MS').strftime('%Y%m%d')
TEST_FORECAST_DATES = pd.date_range('2024-09-01', '2026-03-01', freq='MS').strftime('%Y%m%d')

ERA_VARS = ['sd', 't2m', 'tp', 't2m_min', 't2m_max', 'td2m', 'ts', 'u10', 'v10', 'olr', 'msl']
FOREC_VARS = ['t2m', 'tp', 'ts']
CLIM_VARS = ['sd', 't2m', 'tp']

HIST_LEN = 12
FUT_LEN = 18
LAG = 1

XY_STEP = 4
T_STEP = 7
X0, Y0 = 0, 0
XSIZE, YSIZE = 360, 91

AMP_MIN = 1e-3
SNOW_MM_THR = 5.0
AMP_MAX = 500.0

BATCH_SIZE = 256
SEED = 0

def doy_feat(dates):
    doy = dates.dayofyear.values.astype(np.float32)
    s = torch.tensor(np.sin(2 * np.pi * doy / 365.25), dtype=torch.float32)
    c = torch.tensor(np.cos(2 * np.pi * doy / 365.25), dtype=torch.float32)
    return s, c

def cut(x):
    return x.reshape(x.shape[0], -1)[:, valid_flat]

def check_nan(d, path=''):
    for k, v in d.items():
        p = f'{path}.{k}' if path else k
        if isinstance(v, dict):
            check_nan(v, p)
        elif torch.is_tensor(v):
            n = torch.isnan(v).sum().item()
            if n > 0:
                print(f'{p}: {n} NaN / {v.numel()}')

def gen_dataset(dates, forec=False):
    for date in tqdm(dates):
        t0 = pd.Timestamp(date)
        lag = 0 if forec else LAG
        fut_len = FUT_LEN - (LAG - lag)
        hist_start = t0 - pd.Timedelta(days=7*(HIST_LEN + lag))
        fut_start  = t0 - pd.Timedelta(days=7*lag)

        hist_dates = pd.date_range(hist_start, periods=HIST_LEN, freq='7D')
        fut_dates  = pd.date_range(fut_start,  periods=fut_len,  freq='7D')

        hist_reqs = [patcher.Request(v, X0, Y0, hist_start.strftime('%Y%m%d'),
                                     XSIZE, YSIZE, HIST_LEN, XY_STEP, T_STEP) for v in ERA_VARS]
        hist_clim_reqs = [patcher.Request(v, X0, Y0, hist_start.strftime('%Y%m%d'),
                                     XSIZE, YSIZE, HIST_LEN, XY_STEP, T_STEP) for v in CLIM_VARS]
        fut_reqs  = [patcher.Request(v, X0, Y0, fut_start.strftime('%Y%m%d'),
                                     XSIZE, YSIZE, fut_len, XY_STEP, T_STEP) for v in ERA_VARS]
        fut_clim_reqs = [patcher.Request(v, X0, Y0, fut_start.strftime('%Y%m%d'),
                                     XSIZE, YSIZE, fut_len, XY_STEP, T_STEP) for v in CLIM_VARS]
        fut_forec_reqs  = [patcher.Request('inm_' + v, X0, Y0, fut_start.strftime('%Y%m%d'),
                                     XSIZE, YSIZE, fut_len, XY_STEP // 4, T_STEP, t0.month)
                     for v in FOREC_VARS]

        sd_idx = ERA_VARS.index('sd')
        sd_clim_idx = CLIM_VARS.index('sd')
        hist      = patcher.load(context, hist_reqs)
        hist_clim = patcher.load_climate(context, hist_clim_reqs)
        hist[sd_idx] = hist[sd_idx] + hist_clim[sd_clim_idx]
        fut       = patcher.load(context, fut_reqs)
        fut_clim  = patcher.load_climate(context, fut_clim_reqs)
        fut[sd_idx] = fut[sd_idx] + fut_clim[sd_clim_idx]
        E = 1
        if forec:
            fut_forec = patcher.load(context, fut_forec_reqs)
            E = fut_forec[0].shape[0] + 1
            for i, v in enumerate(FOREC_VARS):
                fut[ERA_VARS.index(v)] = fut_forec[i].mean(axis=0)

        hist_doy_sin, hist_doy_cos = doy_feat(hist_dates)
        fut_doy_sin,  fut_doy_cos  = doy_feat(fut_dates)

        result = []
        for i in range(E):
            member = {
                't0': t0,
                'hist': {
                    'doy':  torch.tensor(hist_dates.dayofyear.values,  dtype=torch.int16),
                    'sin_doy':  hist_doy_sin, 'cos_doy':  hist_doy_cos,
                    'year': torch.tensor(hist_dates.year.values,       dtype=torch.int16),
                },
                'fut': {
                    'doy':   torch.tensor(fut_dates.dayofyear.values,   dtype=torch.int16),
                    'sin_doy':  fut_doy_sin, 'cos_doy':  fut_doy_cos,
                    'year':  torch.tensor(fut_dates.year.values,        dtype=torch.int16),
                },
            }
            for j, v in enumerate(ERA_VARS):
                member['hist'][v] = cut(hist[j])
                member['fut'][v] = cut(fut[j])
            for j, v in enumerate(CLIM_VARS):
                member['hist'][v + '_clim'] = cut(hist_clim[j])
                member['fut'][v + '_clim'] = cut(fut_clim[j])

            sd_window = torch.cat([member['hist']['sd'], member['fut']['sd']], dim=0)
            sd_clim_window = torch.cat([member['hist']['sd_clim'], member['fut']['sd_clim']], dim=0)
            member['fut']['sd_clim_diff'] = torch.diff(sd_clim_window, dim=0)[-FUT_LEN:]

            check_nan(member)
            result.append(member)

            if forec:
                for j, v in enumerate(FOREC_VARS):
                    fut[ERA_VARS.index(v)] = fut_forec[j][i-1]

        if len(result) == 1:
            result = result[0]
        yield result

def build_stats_from_runs(runs):
    stats = {}
    for k in ERA_VARS:
        x = torch.cat([r['hist'][k] for r in runs], dim=1).float()
        stats[k] = (x.mean().item(), x.std().item())
        if k + '_clim' in runs[0]['hist']:
            x = torch.cat([r['hist'][k + '_clim'] for r in runs], dim=1).float()
            stats[k + '_clim'] = (x.mean().item(), x.std().item())

    z_flat = static_data['z'].reshape(-1)[valid_flat].float()
    s_flat = static_data['sdor'].reshape(-1)[valid_flat].float()
    zl = torch.log(z_flat - z_flat.min() + 1)
    sl = torch.log(s_flat - s_flat.min() + 1)
    stats['log_z']    = (zl.mean().item(), zl.std().item())
    stats['log_sdor'] = (sl.mean().item(), sl.std().item())
    return stats

def normalize_run(run, stats):
    for br in ['hist', 'fut']:
        for k in ERA_VARS:
            if k in run[br]:
                m, s = stats[k]
                run[br][k]      = (run[br][k] - m) / s
            if k + '_clim' in run[br]:
                m, s = stats[k + '_clim']
                run[br][k + '_clim'] = (run[br][k + '_clim'] - m) / s
    m, s = stats['sd_clim']
    run['fut']['sd_clim_diff'] = run['fut']['sd_clim_diff'] / s
    return run

def build_batches(runs, stats, seed=0):
    samples = []
    for ri, r in enumerate(runs):
        n = r['hist']['t2m'].shape[1]
        for ni in range(n):
            samples.append((ri, ni))

    rng = np.random.default_rng(seed)
    rng.shuffle(samples)

    batches = []
    for i in tqdm(range(0, len(samples) - BATCH_SIZE + 1, BATCH_SIZE)):
        chunk = samples[i:i + BATCH_SIZE]
        idx_run  = torch.tensor([c[0] for c in chunk])
        idx_node = torch.tensor([c[1] for c in chunk])
        node_ids_b = node_ids[idx_node]

        b = {'t0': [runs[j]['t0'] for j in idx_run]}
        for br in ['hist', 'fut']:
            b[br] = {}
            for k in runs[0][br]:
                if runs[0][br][k].ndim == 1:
                    v = runs[0][br][k]
                    b[br][k] = torch.stack([v for _ in idx_run], dim=0)
                else:
                    b[br][k] = torch.stack([runs[j][br][k][:, n] for j, n in zip(idx_run, idx_node)], dim=0)

        for k in ['z', 'sdor', 'lat', 'lon', 'lat_sin', 'lon_sin', 'lon_cos', 'lat_cos']:
            b[k] = static_data[k][node_ids_b]
        b['node_ids'] = node_ids_b
        check_nan(b)
        batches.append(b)

    return batches

context = patcher.Context('../db', 4)

reqs = [patcher.Request(v, X0, Y0, '19800101', XSIZE, YSIZE, 1, XY_STEP, 1) for v in ['z', 'sdor']]
static_data = patcher.load(context, reqs)
lat = torch.arange(0, 91)
lon = torch.arange(-180, 180)
lat_grid = lat[:, None].expand(YSIZE, XSIZE)
lon_grid = lon[None, :].expand(YSIZE, XSIZE)
static_data = {
    'z': static_data[0][0],
    'sdor': static_data[1][0],
    'lat': lat_grid,
    'lon': lon_grid,
}
static_data['lat_sin'] = torch.sin(torch.deg2rad(lat_grid))
static_data['lat_cos'] = torch.cos(torch.deg2rad(lat_grid))
static_data['lon_sin'] = torch.sin(torch.deg2rad(lon_grid))
static_data['lon_cos'] = torch.cos(torch.deg2rad(lon_grid))

sd_clim = patcher.load_climate(context, [patcher.Request('sd', X0, Y0, '19800101', XSIZE, YSIZE, 52, XY_STEP, T_STEP)])[0]
sd_clim = sd_clim.reshape(52, *sd_clim.shape[-2:])
sd_clim_mean = sd_clim.mean(0)
valid = (sd_clim_mean > AMP_MIN) & (sd_clim_mean < AMP_MAX)
valid_flat = valid.reshape(-1)
node_ids   = torch.nonzero(valid_flat, as_tuple=False).squeeze(-1)

train_runs = list(gen_dataset(TRAIN_DATES))
stats = build_stats_from_runs(train_runs)
static_data['z'] = (torch.log(static_data['z'] - static_data['z'].min() + 1) - stats['log_z'][0]) / stats['log_z'][1]
static_data['sdor'] = (torch.log(static_data['sdor'] - static_data['sdor'].min() + 1) - stats['log_sdor'][0]) / stats['log_sdor'][1]
for k in static_data:
    static_data[k] = static_data[k].ravel()
train_runs = [normalize_run(run, stats) for run in train_runs]
train_batches = build_batches(train_runs, stats)
torch.save(train_batches, f'{BASE_PATH}/train_batches.pt')
del train_runs
del train_batches

test_runs = [normalize_run(run, stats) for run in gen_dataset(TEST_DATES)]
test_batches = build_batches(test_runs, stats)
torch.save(test_batches, f'{BASE_PATH}/test_batches.pt')
del test_runs
del test_batches

static_data['valid_flat'] = valid_flat
static_data['node_ids'] = node_ids
torch.save(stats, f'{BASE_PATH}/stats.pt')
torch.save(static_data, f'{BASE_PATH}/static.pt')

runs = []
for run in gen_dataset(TEST_FORECAST_DATES, forec=True):
    if len(runs) == 0:
        for i in range(len(run)):
            runs.append([normalize_run(run[i], stats)])
    else:
        for i in range(len(run)):
            runs[i].append(normalize_run(run[i], stats))
    
for i in range(len(runs)):
    name = 'test_batches_forec_mean' if i == 0 else f'test_batches_forec_e{i:02d}'
    test_batches = build_batches(runs[i], stats)
    torch.save(test_batches, f'{BASE_PATH}/{name}.pt')

del runs
del test_batches

In [2]:
train_ds = torch.load(f'{BASE_PATH}/train_batches.pt', weights_only=False)
test_ds  = torch.load(f'{BASE_PATH}/test_batches.pt',  weights_only=False)
stats = torch.load(f'{BASE_PATH}/stats.pt', weights_only=False)
static = torch.load(f'{BASE_PATH}/static.pt', weights_only=False)

# Модель

In [3]:
class ClimatologyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.dummy = nn.Parameter(torch.zeros(1))

    def forward(self, batch):
        pred = batch['fut']['sd_clim']
        snow_logits = torch.zeros_like(pred)
        return pred + self.dummy * 0, snow_logits

class PersistenceAnomalyModel(nn.Module):
    def __init__(self, min_swe=0.0):
        super().__init__()
        self.min_swe = min_swe
        self.dummy = nn.Parameter(torch.zeros(1))

    def forward(self, batch):
        sd_last      = batch['hist']['sd'][:, -1]
        sd_clim_hist = batch['hist']['sd_clim'][:, -1]

        anom = sd_last - sd_clim_hist
        sd_clim_fut = batch['fut']['sd_clim']
        pred = torch.clamp(sd_clim_fut + anom.unsqueeze(1), min=self.min_swe)
        snow_logits = torch.zeros_like(pred)
        return pred + self.dummy*0, snow_logits

class SnowMoE(nn.Module):
    def __init__(self, hist_vars, fut_vars, hidden=16, n_layers_hist=2, n_layers_expert=2, dropout=0.1, k_experts=20, top_k=2, min_swe=0.0):
        super().__init__()
        self.hist_vars = hist_vars
        self.fut_vars = fut_vars
        self.hidden = hidden
        self.k_experts = k_experts
        self.top_k = top_k
        self.min_swe = min_swe
        self.n_layers_hist = n_layers_hist
        self.n_layers_expert = n_layers_expert

        self.hist_gru = nn.GRU(
            input_size=len(hist_vars),
            hidden_size=hidden,
            num_layers=n_layers_hist,
            batch_first=True,
            dropout=dropout if n_layers_hist > 1 else 0.0,
        )

        self.static_mlp = nn.Sequential(*(
            [nn.Linear(6, hidden), nn.ReLU()] +
            [l for _ in range(n_layers_hist - 1)
               for l in (nn.Linear(hidden, hidden), nn.ReLU())]
        ))

        self.gate = nn.Sequential(
            nn.Linear(6, hidden),
            nn.ReLU(),
            nn.Linear(hidden, k_experts),
        )

        self.experts = nn.ModuleList([
            nn.Sequential(*(
                [nn.Linear(hidden + 2 + len(fut_vars), hidden), nn.ReLU()] +
                [l for _ in range(n_layers_expert - 1)
                   for l in (nn.Linear(hidden, hidden), nn.ReLU())]
            ))
            for _ in range(k_experts)
        ])

        self.head_snowfall = nn.Linear(hidden, 1)
        self.head_melt = nn.Linear(hidden, 1)
        self.head_snowcover = nn.Linear(hidden, 1)

        nn.init.constant_(self.head_snowfall.bias, -2.0)
        nn.init.constant_(self.head_melt.bias, -2.0)
        nn.init.zeros_(self.head_snowfall.weight)
        nn.init.zeros_(self.head_melt.weight)

    def get_weights(self, fut):
        sd_clim = fut['sd_clim']
        clim_stats = torch.stack([
            sd_clim.mean(1),
            sd_clim.std(1),
            sd_clim.max(1).values,
            sd_clim[:, -1] - sd_clim[:, 0],
            sd_clim.argmax(1).float(),
            (sd_clim > 1e-3).float().mean(1),
        ], dim=-1)

        w = torch.softmax(self.gate(clim_stats), dim=-1)
        topk_w, topk_idx = w.topk(self.top_k, dim=-1)
        topk_w = topk_w / (topk_w.sum(-1, keepdim=True) + 1e-6)
        return topk_w, topk_idx, w
    
    def forward(self, batch):
        hist = batch['hist']
        fut = batch['fut']

        B = hist['sd'].shape[0]

        hist_x = torch.stack([hist[k] for k in self.hist_vars], dim=-1)
        _, h = self.hist_gru(hist_x)
        h_dyn = h[-1]

        static_x = torch.stack([
            batch['z'], batch['sdor'],
            batch['lat_sin'], batch['lat_cos'],
            batch['lon_sin'], batch['lon_cos'],
        ], dim=-1)
        h_stat = self.static_mlp(static_x)

        h_ctx = h_dyn + h_stat

        topk_w, topk_idx, w = self.get_weights(fut)
        unique_experts = topk_idx.unique()

        swe = hist['sd'][:, -1]
        T = fut[self.fut_vars[0]].shape[1]
        preds, snow_logits = [], []
       
        for k in range(T):
            z_k = torch.cat([h_ctx, swe[:, None], torch.full((B, 1), k / T, device=swe.device)] +
                [fut[v][:, k:k+1] for v in self.fut_vars], dim=-1)

            z_all = torch.zeros(B, self.k_experts, self.hidden, device=w.device)
            for e in unique_experts:
                mask = (topk_idx == e).any(dim=1)
                z_all[mask, e] = self.experts[e.item()](z_k[mask])
            z_sel = torch.gather(
                z_all, 1,
                topk_idx.unsqueeze(-1).expand(-1, -1, self.hidden)
            )
            z_final = (topk_w.unsqueeze(-1) * z_sel).sum(1)

            snowfall = F.softplus(self.head_snowfall(z_final).squeeze(-1))
            melt = F.softplus(self.head_melt(z_final).squeeze(-1))
            snowcover = self.head_snowcover(z_final).squeeze(-1)
           
            dSWE = snowfall - melt
            swe = torch.clamp(swe + dSWE, min=self.min_swe)

            preds.append(swe)
            snow_logits.append(snowcover)

        frac_used = torch.zeros(self.k_experts, device=w.device)
        for e in range(self.k_experts):
            frac_used[e] = (topk_idx == e).float().mean()
        loss_bal = self.k_experts * (frac_used * w.mean(0)).sum()
        
        return torch.stack(preds, dim=1), torch.stack(snow_logits, dim=1), loss_bal

def save_model(path, model, optimizer, info=None):
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if optimizer is not None else None,
        'info': info or {}
    }
    torch.save(checkpoint, path)

def load_model(path, model, optimizer=None, map_location='cpu'):
    checkpoint = torch.load(path, map_location=map_location)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    
    if optimizer is not None and checkpoint.get('optimizer_state_dict') is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    return checkpoint.get('info', {})

# Обучение

In [4]:
def symlog(x):
    return torch.sign(x) * torch.log1p(torch.abs(x))

def loss_fn(pred, snow_logits, loss_bal, target, eps=1e-3, w_reg=1.0, w_cls=0.5, w_bal=0.01):
    log_mse = (symlog(pred) - symlog(target)) ** 2

    loss_reg = log_mse.mean()

    snow_true = (target > eps).float()
    loss_cls = F.binary_cross_entropy_with_logits(snow_logits, snow_true)

    return w_reg * loss_reg + w_cls * loss_cls + w_bal * loss_bal

def to_mm(x, stats):
    return x * stats['sd'][1] + stats['sd'][0]

def train_cycle(name, model, train, test, optimizer, loss_fn, epochs, stats):
    cur_epoch = 0
    model_path = f'{BASE_MODEL_PATH}/{name}.pt'
    train_loss_hist, test_loss_hist, test_loss_mm_hist, acc_hist = [], [], [], []
    if os.path.exists(model_path):
        info = load_model(model_path, model, optimizer)
        cur_epoch = info['epoch']
        train_loss_hist = info['train_loss_hist']
        test_loss_hist = info['test_loss_hist']
        test_loss_mm_hist = info['test_loss_mm_hist']
        acc_hist = info['acc_hist']
    for epoch in range(cur_epoch, epochs):
        model.train()
        train_loss = 0.0
        for batch in tqdm(train):
            y = batch['fut']['sd']
            pred, snow_logits, loss_bal = model(batch)
            loss = loss_fn(pred, snow_logits, loss_bal, y)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            train_loss += loss.item()

        model.eval()
        test_loss = 0.0
        test_loss_mm = 0.0
        pred_anom, true_anom = [], []
        n = 0
        with torch.no_grad():
            for batch in tqdm(test):
                y = batch['fut']['sd']
                pred, snow_logits, loss_bal = model(batch)
                test_loss += loss_fn(pred, snow_logits, loss_bal, y).item()

                test_loss_mm += ((to_mm(pred, stats) - to_mm(y, stats)) ** 2).sum().item()
                n += pred.numel()

                clim = batch['fut']['sd_clim']
                pred_anom.append((pred - clim).flatten())
                true_anom.append((y - clim).flatten())

        train_loss /= len(train)
        test_loss /= len(test)
        test_loss_mm = (test_loss_mm / n) ** 0.5
        pred_anom, true_anom = torch.cat(pred_anom), torch.cat(true_anom)
        acc = ((pred_anom * true_anom).sum() / (pred_anom.pow(2).sum().sqrt() * true_anom.pow(2).sum().sqrt() + 1e-6)).item()

        if epoch + 1 != epochs:
            print(f'epoch {epoch + 1}: train={train_loss:.4f}  test={test_loss:.4f}  '
                  f'test_rmse_mm={test_loss_mm:.2f}  ACC={acc:.3f}')
        train_loss_hist.append(train_loss)
        test_loss_hist.append(test_loss)
        test_loss_mm_hist.append(test_loss_mm)
        acc_hist.append(acc)
        info = {
            'epoch': epoch + 1,
            'train_loss_hist': train_loss_hist,
            'test_loss_hist': test_loss_hist,
            'test_loss_mm_hist': test_loss_mm_hist,
            'acc_hist': acc_hist,
        }
        save_model(model_path, model, optimizer, info)
    print(f'epoch {epochs}: train={train_loss_hist[-1]:.4f}  test={test_loss_hist[-1]:.4f}  '
            f'test_rmse_mm={test_loss_mm_hist[-1]:.2f}  ACC={acc_hist[-1]:.3f}')

# Анализ

In [5]:
def clone_batch(batch):
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.clone()
        elif isinstance(v, dict):
            out[k] = {kk: (vv.clone() if torch.is_tensor(vv) else vv)
                      for kk, vv in v.items()}
        else:
            out[k] = v
    return out


def evaluate_rmse(model, ds, sd_std, ablate_branch=None, ablate_field=None):
    model.eval()
    total_sq, n = 0.0, 0
    with torch.no_grad():
        for i, batch in enumerate(ds):
            if ablate_branch is not None:
                batch = clone_batch(batch)
                x = batch[ablate_branch][ablate_field]
                batch[ablate_branch][ablate_field] = x.mean(dim=0, keepdim=True).expand_as(x)
            y = batch['fut']['sd']
            pred, _, _ = model(batch)
            d = (pred - y) * sd_std
            total_sq += (d ** 2).sum().item()
            n += d.numel()

    return (total_sq / n) ** 0.5


def run_ablation(model, ds, sd_std):
    baseline = evaluate_rmse(model, ds, sd_std)
    print(f'\nbaseline RMSE: {baseline:.3f} mm\n')
    print(f'{"field":<20} {"RMSE":>8} {"ΔRMSE":>8}')

    fields = [('hist', v) for v in model.hist_vars] + [('fut', v) for v in model.fut_vars]

    results = []
    for branch, field in fields:
        rmse = evaluate_rmse(model, ds, sd_std, ablate_branch=branch, ablate_field=field)
        delta = rmse - baseline
        results.append((f'{branch}.{field}', rmse, delta))
        print(f'{branch+"."+field:<20} {rmse:>8.3f} {delta:>+8.3f}')

    results.sort(key=lambda r: r[2], reverse=True)
    print('\nТоп по вкладу:')
    for name, rmse, delta in results:
        print(f'  {name:<20} Δ={delta:+.3f}')

In [6]:
def find_node_id(lat_target, lon_target, static):
    lat_grid = static['lat']
    lon_grid = static['lon']
    d = (lat_grid - lat_target) ** 2 + (lon_grid - lon_target) ** 2
    return d.argmin()

def build_index(batches):
    idx = {}
    for bi, batch in enumerate(batches):
        for i, t in enumerate(batch['t0']):
            key = (pd.Timestamp(t).normalize(), int(batch['node_ids'][i]))
            idx[key] = (bi, i)
    return idx

def _slice_batch(batch, i, key):
    v = batch[key]
    if torch.is_tensor(v):
        return v[i:i+1]
    if isinstance(v, dict):
        return {k: _slice_batch(v, i, k) for k in v}
    return v

def plot_run(model, ds, index, node_id, t0_target, stats, filename=None):
    model.eval()
    t0_target = pd.Timestamp(t0_target)
    bi, i = index[(t0_target.normalize(), node_id)]

    batch = ds[bi]
    pred, _, _ = model({k: _slice_batch(batch, i, k) for k in batch if k != 't0'})
    pred = pred.detach().numpy()

    sd_mean, sd_std = stats['sd']
    sd_clim_mean, sd_clim_std = stats['sd_clim']
    pred_mm = (pred * sd_std + sd_mean).ravel()
    true_mm = np.concatenate([
        (batch['hist']['sd'][i] * sd_std + sd_mean).numpy().ravel(),
        (batch['fut']['sd'][i] * sd_std + sd_mean).numpy().ravel(),
    ])

    clim_mm = np.concatenate([
        (batch['hist']['sd_clim'][i] * sd_clim_std + sd_clim_mean).numpy().ravel(),
        (batch['fut']['sd_clim'][i] * sd_clim_std + sd_clim_mean).numpy().ravel(),
    ])

    hist_weeks = -np.arange(len(batch['hist']['sd'][i]), 0, -1)
    fut_weeks = np.arange(0, len(batch['fut']['sd'][i]))
    weeks = np.concatenate([hist_weeks, fut_weeks])
    
    t2m = np.concatenate([
        batch['hist']['t2m'][i].numpy() * stats['t2m'][1] + stats['t2m'][0] +
        batch['hist']['t2m_clim'][i].numpy() * stats['t2m_clim'][1] + stats['t2m_clim'][0],
        batch['fut']['t2m'][i].numpy()  * stats['t2m'][1] + stats['t2m'][0] +
        batch['fut']['t2m_clim'][i].numpy() * stats['t2m_clim'][1] + stats['t2m_clim'][0],
    ]) - 273.15

    tp = np.concatenate([
        batch['hist']['tp'][i].numpy() * stats['tp'][1] + stats['tp'][0] +
        batch['hist']['tp_clim'][i].numpy() * stats['tp_clim'][1] + stats['tp_clim'][0],
        batch['fut']['tp'][i].numpy()  * stats['tp'][1] + stats['tp'][0] +
        batch['fut']['tp_clim'][i].numpy() * stats['tp_clim'][1] + stats['tp_clim'][0],
    ])

    lat = float(batch['lat'][i])
    lon = float(batch['lon'][i])
    node = int(batch['node_ids'][i])

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8),
                                   gridspec_kw={'height_ratios': [2, 1]},
                                   sharex=True)

    ax1.plot(weeks, true_mm, 'k-o', ms=4, label='true sd')
    ax1.plot(weeks, clim_mm, 'k--o', ms=4, label='sd clim')
    ax1.plot(fut_weeks, pred_mm, 'r--o', ms=4, label='pred sd')
    ax1.axvline(0, color='gray', lw=0.8, ls=':')
    ax1.set_ylabel('sd, mm')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.set_title(
        f't0={t0_target.date()}  node={node}  lat={lat:.1f} lon={lon:.1f}'
    )
    ax1.set_xticks(list(hist_weeks) + list(fut_weeks))

    ax2.plot(weeks, t2m, 'b-o', ms=3, label='t2m')
    ax2.set_ylabel('t2m, °C', color='b')
    ax2.tick_params(axis='y', labelcolor='b')
    ax2.axhline(0, color='b', lw=0.5, ls=':')
    ax2.grid(True, alpha=0.3)

    ax2b = ax2.twinx()
    ax2b.bar(weeks, tp, width=0.7, color='g', alpha=0.6)
    ax2b.set_ylabel('tp, mm', color='g')
    ax2b.tick_params(axis='y', labelcolor='g')

    ax2.axvline(0, color='gray', lw=0.8, ls=':')
    ax2.set_xlabel('week relative to t0')

    lines1, labels1 = ax2.get_legend_handles_labels()
    lines2, labels2 = ax2b.get_legend_handles_labels()
    ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

    plt.tight_layout()
    if filename is None:
        plt.show()
    else:
        plt.savefig(filename, dpi=150)
        plt.close()

In [7]:
def plot_experts(model, ds, index, t0, static, filename=None):
    model.eval()
    t0 = pd.Timestamp(t0).normalize()

    node_ids, dom = [], []
    with torch.no_grad():
        for (t, nid), (bi, i) in index.items():
            if t != t0:
                continue
            batch = ds[bi]
            sub = {k: _slice_batch(batch, i, k) for k in batch if k != 't0'}
            topk_w, topk_idx, _ = model.get_weights(sub['fut'])
            topk_w   = topk_w.cpu().numpy()
            topk_idx = topk_idx.cpu().numpy()
            d = topk_idx[0, topk_w[0].argmax()]
            node_ids.append(nid)
            dom.append(d)

    node_ids = np.array(node_ids)
    dom      = np.array(dom)

    lat = static['lat'].reshape((91, 360))[:, 0].numpy()
    lon = static['lon'].reshape((91, 360))[0, :].numpy()
    nlat, nlon = len(lat), len(lon)

    grid = np.full((nlat, nlon), np.nan, dtype=np.float32)
    rows, cols = np.divmod(node_ids, nlon)
    grid[rows, cols] = dom

    K = model.k_experts
    c1 = plt.get_cmap('tab20')(np.linspace(0, 1, 20))
    c2 = plt.get_cmap('tab20b')(np.linspace(0, 1, 20))
    cmap = plt.matplotlib.colors.ListedColormap(np.vstack([c1, c2])[:K])

    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(grid, cmap=cmap, vmin=-0.5, vmax=K - 0.5,
                   origin='lower', aspect='auto',
                   extent=[lon[0], lon[-1], lat[0], lat[-1]])
    fig.colorbar(im, ax=ax, ticks=np.arange(K)).set_label('expert id')
    ax.set_xlabel('lon'); ax.set_ylabel('lat')
    ax.set_title(f'dominant expert, t0 = {t0:%Y-%m-%d}')

    if filename:
        fig.savefig(filename, dpi=150, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()

In [11]:
def save_model_predictions(model, ds, stats, static, path):
    model.eval()

    sd_mean, sd_std = stats['sd']
    sd_clim_mean, sd_clim_std = stats['sd_clim']
    valid_flat = static['valid_flat']
    node_ids = static['node_ids']
    lat = static['lat'].reshape((91, 360))[:, 0]
    lon = static['lon'].reshape((91, 360))[0, :]
    nlat, nlon = len(lat), len(lon)

    directory = Path(path)
    directory.mkdir(parents=True, exist_ok=True)

    runs = defaultdict(dict)
    for batch in tqdm(ds):
        with torch.no_grad():
            preds, _, _ = model(batch)
        preds = preds * sd_std + sd_mean
        clim = batch['fut']['sd_clim'] * sd_clim_std + sd_clim_mean
        anom = (preds - clim).cpu().numpy()

        B, T = anom.shape
        for i in range(B):
            t0 = batch['t0'][i]
            nid = batch['node_ids'][i].item()
            runs[t0][nid] = anom[i][1:]

    for t0, node_vals in tqdm(runs.items()):
        t0 = pd.Timestamp(t0)
        T = 17
        grid = np.full((T, nlat*nlon), np.nan, dtype=np.float32)
        for nid, vals in node_vals.items():
            grid[:, nid] = vals

        times = np.array([t0 + timedelta(days=7 * k) for k in range(T)], dtype='datetime64[D]')
        grid = grid.reshape((T, nlat, nlon))

        ds = xr.Dataset(
            data_vars={
                'swe_anom_weekly': (('time_weekly', 'lat', 'lon'), grid)
            },
            coords={
                'time_weekly': times,
                'lat': lat,
                'lon': lon,
            }
        )
        out = directory / f'{t0.strftime("%Y%m")}.nc'
        ds.to_netcdf(out, engine='h5netcdf',
            encoding={v: {'zlib': True, 'complevel': 4, 'shuffle': True} for v in ds.data_vars}
        )

# Эксперименты

In [10]:
model = ClimatologyModel()
opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
train_cycle('climate', model, train_ds, test_ds, opt, loss_fn, 1, stats)

model = PersistenceAnomalyModel(min_swe=-stats['sd'][0] / stats['sd'][1])
opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
train_cycle('persistance', model, train_ds, test_ds, opt, loss_fn, 1, stats)

epoch 1: train=0.3811  test=0.3841  test_rmse_mm=21.39  ACC=0.000
epoch 1: train=0.3838  test=0.3772  test_rmse_mm=17.25  ACC=0.609


In [5]:
hist_vars = ['sd', 't2m', 'tp', 'ts', 't2m_min', 't2m_max', 'msl', 'olr', 'u10', 'v10',
             't2m_clim', 'tp_clim', 'sd_clim', 'sin_doy', 'cos_doy']
fut_vars = hist_vars[1:] + ['sd_clim_diff']
model = SnowMoE(hist_vars, fut_vars, hidden=32, k_experts=1, top_k=1, min_swe=-stats['sd'][0] / stats['sd'][1])
opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
train_cycle('all_vars', model, train_ds, test_ds, opt, loss_fn, 5, stats)

  0%|          | 0/18319 [00:00<?, ?it/s]

  0%|          | 0/2181 [00:00<?, ?it/s]

epoch 1: train=0.0890  test=0.0686  test_rmse_mm=11.99  ACC=0.830


  0%|          | 0/18319 [00:00<?, ?it/s]

  0%|          | 0/2181 [00:00<?, ?it/s]

...

  0%|          | 0/2181 [00:00<?, ?it/s]

epoch 4: train=0.0621  test=0.0638  test_rmse_mm=11.08  ACC=0.856


  0%|          | 0/18319 [00:00<?, ?it/s]

  0%|          | 0/2181 [00:00<?, ?it/s]

epoch 5: train=0.0614  test=0.0630  test_rmse_mm=10.95  ACC=0.860


In [10]:
hist_vars = ['sd', 't2m', 'tp', 'ts', 't2m_min', 't2m_max', 'msl', 'olr', 'u10', 'v10',
             't2m_clim', 'tp_clim', 'sd_clim', 'sin_doy', 'cos_doy']
fut_vars = hist_vars[1:] + ['sd_clim_diff']
model = SnowMoE(hist_vars, fut_vars, hidden=32, k_experts=1, top_k=1, min_swe=-stats['sd'][0] / stats['sd'][1])
opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
train_cycle('all_vars', model, train_ds, test_ds, opt, loss_fn, 10, stats)

  0%|          | 0/18319 [00:00<?, ?it/s]

  0%|          | 0/2181 [00:00<?, ?it/s]

epoch 6: train=0.0609  test=0.0625  test_rmse_mm=10.89  ACC=0.861


  0%|          | 0/18319 [00:00<?, ?it/s]

  0%|          | 0/2181 [00:00<?, ?it/s]

...

  0%|          | 0/2181 [00:00<?, ?it/s]

epoch 9: train=0.0598  test=0.0613  test_rmse_mm=10.73  ACC=0.866


  0%|          | 0/18319 [00:00<?, ?it/s]

  0%|          | 0/2181 [00:00<?, ?it/s]

epoch 10: train=0.0596  test=0.0610  test_rmse_mm=10.69  ACC=0.867


In [28]:
hist_vars = ['sd', 't2m', 'tp', 'ts', 't2m_min', 't2m_max', 'msl', 'olr', 'u10', 'v10',
             't2m_clim', 'tp_clim', 'sd_clim', 'sin_doy', 'cos_doy']
fut_vars = hist_vars[1:] + ['sd_clim_diff']
model = SnowMoE(hist_vars, fut_vars, hidden=32, k_experts=1, top_k=1, min_swe=-stats['sd'][0] / stats['sd'][1])
opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
train_cycle('all_vars', model, train_ds, test_ds, opt, loss_fn, 15, stats)

  0%|          | 0/18319 [00:00<?, ?it/s]

  0%|          | 0/2181 [00:00<?, ?it/s]

epoch 11: train=0.0494  test=0.0508  test_rmse_mm=10.66  ACC=0.868


  0%|          | 0/18319 [00:00<?, ?it/s]

  0%|          | 0/2181 [00:00<?, ?it/s]

...

  0%|          | 0/2181 [00:00<?, ?it/s]

epoch 14: train=0.0489  test=0.0504  test_rmse_mm=10.59  ACC=0.869


  0%|          | 0/18319 [00:00<?, ?it/s]

  0%|          | 0/2181 [00:00<?, ?it/s]

epoch 15: train=0.0487  test=0.0503  test_rmse_mm=10.56  ACC=0.870


In [6]:
hist_vars = ['sd', 't2m', 'tp', 'ts', 't2m_min', 't2m_max', 'msl', 'olr', 'u10', 'v10',
             't2m_clim', 'tp_clim', 'sd_clim', 'sin_doy', 'cos_doy']
fut_vars = hist_vars[1:] + ['sd_clim_diff']
model = SnowMoE(hist_vars, fut_vars, hidden=32, k_experts=32, top_k=2, min_swe=-stats['sd'][0] / stats['sd'][1])
opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
train_cycle('all_vars_experts', model, train_ds, test_ds, opt, loss_fn, 5, stats)

  0%|          | 0/18319 [00:00<?, ?it/s]

  0%|          | 0/2181 [00:00<?, ?it/s]

epoch 1: train=0.0945  test=0.0708  test_rmse_mm=12.18  ACC=0.824


  0%|          | 0/18319 [00:00<?, ?it/s]

  0%|          | 0/2181 [00:00<?, ?it/s]

...

  0%|          | 0/2181 [00:00<?, ?it/s]

epoch 4: train=0.0646  test=0.0645  test_rmse_mm=11.23  ACC=0.853


  0%|          | 0/18319 [00:00<?, ?it/s]

  0%|          | 0/2181 [00:00<?, ?it/s]

epoch 5: train=0.0637  test=0.0642  test_rmse_mm=11.08  ACC=0.857


In [11]:
hist_vars = ['sd', 't2m', 'tp', 'ts', 't2m_min', 't2m_max', 'msl', 'olr', 'u10', 'v10',
             't2m_clim', 'tp_clim', 'sd_clim', 'sin_doy', 'cos_doy']
fut_vars = hist_vars[1:] + ['sd_clim_diff']
model = SnowMoE(hist_vars, fut_vars, hidden=32, k_experts=32, top_k=2, min_swe=-stats['sd'][0] / stats['sd'][1])
opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
train_cycle('all_vars_experts', model, train_ds, test_ds, opt, loss_fn, 10, stats)

  0%|          | 0/18319 [00:00<?, ?it/s]

  0%|          | 0/2181 [00:00<?, ?it/s]

epoch 6: train=0.0632  test=0.0642  test_rmse_mm=11.11  ACC=0.856


  0%|          | 0/18319 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [32]:
hist_vars = ['sd', 't2m', 'tp', 'ts', 't2m_min', 't2m_max', 'msl', 'olr', 'u10', 'v10',
             't2m_clim', 'tp_clim', 'sd_clim', 'sin_doy', 'cos_doy']
fut_vars = hist_vars[1:] + ['sd_clim_diff']
model = SnowMoE(hist_vars, fut_vars, hidden=32, k_experts=16, top_k=2, min_swe=-stats['sd'][0] / stats['sd'][1])
opt = torch.optim.Adam(model.parameters(), lr=2e-4, weight_decay=1e-4)
train_cycle('all_vars_experts2', model, train_ds, test_ds, opt, loss_fn, 6, stats)

epoch 6: train=0.0491  test=0.0507  test_rmse_mm=10.48  ACC=0.872


In [27]:
run_ablation(model, test_ds[:500], stats['sd'][1])


baseline RMSE: 10.500 mm

field                    RMSE    ΔRMSE
hist.sd                36.046  +25.546
hist.t2m               10.898   +0.398
hist.tp                10.533   +0.033
hist.ts                10.474   -0.026
hist.t2m_min           10.501   +0.000
hist.t2m_max           10.519   +0.018
hist.msl               10.503   +0.003
hist.olr               10.476   -0.024
hist.u10               10.501   +0.001
hist.v10               10.502   +0.002
hist.t2m_clim          11.107   +0.606
hist.tp_clim           10.638   +0.137
hist.sd_clim           16.862   +6.362
hist.sin_doy           10.497   -0.004
hist.cos_doy           10.497   -0.003
fut.t2m                14.673   +4.173
fut.tp                 12.883   +2.383
fut.ts                 14.301   +3.801
fut.t2m_min            10.940   +0.440
fut.t2m_max            14.279   +3.779
fut.msl                10.532   +0.032
fut.olr                10.501   +0.000
fut.u10                10.505   +0.005
fut.v10                10.544   +0.04

In [30]:
test_index = build_index(test_ds)
train_index = build_index(train_ds)

In [33]:
train_dates = pd.date_range('1980-06-01', '2020-12-01', freq='MS')
test_dates = pd.date_range('2021-06-01', '2026-03-01', freq='MS')

lat_msk = 55.75
lon_msk = 37.62
msk_node_id = int(find_node_id(lat_msk, lon_msk, static))

for d in tqdm(train_dates):
    plot_run(model, train_ds, train_index, msk_node_id, d.strftime('%Y-%m-%d'), stats, f'../images/msk_era/{d.strftime('%Y%m%d')}.png')
for d in tqdm(test_dates):
    plot_run(model, test_ds, test_index, msk_node_id, d.strftime('%Y-%m-%d'), stats, f'../images/msk_era/{d.strftime('%Y%m%d')}.png')

  0%|          | 0/487 [00:00<?, ?it/s]

  0%|          | 0/58 [00:00<?, ?it/s]

In [34]:
for month in range(1, 13):
    plot_experts(model, test_ds, test_index, f'2025-{month:02d}-01', static, f'../images/experts/{month:02d}.png')

In [9]:
hist_vars = ['sd', 't2m', 'tp', 'ts', 't2m_min', 't2m_max', 'msl', 'olr', 'u10', 'v10',
             't2m_clim', 'tp_clim', 'sd_clim', 'sin_doy', 'cos_doy']
fut_vars = ['t2m', 'tp', 'ts', 't2m_clim', 'tp_clim', 'sd_clim', 'sd_clim_diff', 'sin_doy', 'cos_doy']
model = SnowMoE(hist_vars, fut_vars, hidden=32, k_experts=1, top_k=1, min_swe=-stats['sd'][0] / stats['sd'][1])
opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
train_cycle('forec_vars_experts', model, train_ds, test_ds, opt, loss_fn, 5, stats)

  0%|          | 0/18319 [00:00<?, ?it/s]

  0%|          | 0/2181 [00:00<?, ?it/s]

epoch 1: train=0.0911  test=0.0689  test_rmse_mm=12.03  ACC=0.830


  0%|          | 0/18319 [00:00<?, ?it/s]

  0%|          | 0/2181 [00:00<?, ?it/s]

...

  0%|          | 0/2181 [00:00<?, ?it/s]

epoch 4: train=0.0631  test=0.0653  test_rmse_mm=11.22  ACC=0.852


  0%|          | 0/18319 [00:00<?, ?it/s]

  0%|          | 0/2181 [00:00<?, ?it/s]

epoch 5: train=0.0625  test=0.0645  test_rmse_mm=11.09  ACC=0.855


In [9]:
hist_vars = ['sd', 't2m', 'tp', 'ts', 't2m_min', 't2m_max', 'msl', 'olr', 'u10', 'v10',
             't2m_clim', 'tp_clim', 'sd_clim', 'sin_doy', 'cos_doy']
fut_vars = ['t2m', 'tp', 'ts', 't2m_clim', 'tp_clim', 'sd_clim', 'sd_clim_diff', 'sin_doy', 'cos_doy']
model = SnowMoE(hist_vars, fut_vars, hidden=32, k_experts=1, top_k=1, min_swe=-stats['sd'][0] / stats['sd'][1])
opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
train_cycle('forec_vars_experts', model, train_ds, test_ds, opt, loss_fn, 8, stats)

epoch 8: train=0.0513  test=0.0530  test_rmse_mm=10.81  ACC=0.863


In [12]:
save_model_predictions(model, test_ds, stats, static, f'{BASE_SCORE_PATH}/model_era')

  0%|          | 0/2181 [00:00<?, ?it/s]

  0%|          | 0/58 [00:00<?, ?it/s]

In [56]:
for i in range(31):
    name = 'mean' if i == 0 else f'e{i:02d}'
    test_forec_ds  = torch.load(f'{BASE_PATH}/test_batches_forec_{name}.pt',  weights_only=False)
    save_model_predictions(model, test_forec_ds, stats, static, f'{BASE_SCORE_PATH}/model_{name}')

  0%|          | 0/714 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

  0%|          | 0/714 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

  0%|          | 0/714 [00:00<?, ?it/s]

...

  0%|          | 0/19 [00:00<?, ?it/s]

  0%|          | 0/714 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

  0%|          | 0/714 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

In [57]:
ens_members = [f'model_e{i:02d}' for i in range(1, 31)]
ens_path = Path(f'{BASE_SCORE_PATH}/model_ens')
ens_path.mkdir(parents=True, exist_ok=True)

file_names = sorted(f.name for f in Path(f'{BASE_SCORE_PATH}/{ens_members[0]}').glob('*.nc'))
for fname in tqdm(file_names):
    arrays = []
    template = None
    for m in ens_members:
        fpath = Path(f'{BASE_SCORE_PATH}/{m}/{fname}')
        ds = xr.open_dataset(fpath, engine='h5netcdf')
        arrays.append(ds['swe_anom_weekly'].values)
        if template is None:
            template = ds
        ds.close()
    stack = np.stack(arrays, axis=0)
    valid = ~np.isnan(stack)
    counts = valid.sum(axis=0)
    sums = np.nansum(stack, axis=0)
    with np.errstate(invalid='ignore', divide='ignore'):
        mean_vals = sums / counts
    mean_vals[counts == 0] = np.nan
    out_ds = xr.Dataset(
        data_vars={'swe_anom_weekly': (('time_weekly', 'lat', 'lon'), mean_vals)},
        coords={
            'time_weekly': template['time_weekly'].values,
            'lat': template['lat'].values,
            'lon': template['lon'].values,
        }
    )
    out_ds.to_netcdf(ens_path / fname, engine='h5netcdf',
        encoding={v: {'zlib': True, 'complevel': 4, 'shuffle': True} for v in out_ds.data_vars})

  0%|          | 0/19 [00:00<?, ?it/s]

163421
163523
163693
163523
163540
163591
163574
163472
163489
163557
163523
163540
163540
163506
163659
163489
163591
163506
163591
